# Import Mount Isa from OpenStreetMap

This demo keeps two local input options. The default uses OSMnx to pre-download and cache OSM responses in python/data/osm/osmnx_cache, then AequilibraE imports from that cache. After the first download, reuse or share that directory for a fast repeatable demo. The alternate option imports a prepared .osm.pbf file through pyrosm.

For the OSMnx cache option, no third-party extract service is used: OSMnx fetches the compact Mount Isa study area from OpenStreetMap services. The saved cache contains OSM HTTP response JSON, not a .osm.pbf file. Keep the cache directory and use the same study area, OSMnx version, and request settings when replaying it.

The live area is centered on the top-ranked Mount Isa town result with a 5 km radius, avoiding the much larger administrative boundaries returned by a full place-name search. OSM data is community maintained; retain attribution and review coverage before analysis.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import pandas as pd

from aequilibrae import Project


# A fresh folder lets the notebook be run more than once without replacing an existing project.
project_path = Path(gettempdir()) / f"aequilibrae_mount_isa_{uuid4().hex[:8]}"
print(f"AequilibraE project folder: {project_path}")


meson-python: building aequilibrae: /home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/.venv/bin/ninja
[1/3] Compiling Cython source /home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/src/aequilibrae/paths/cython/route_choice_set_results.pyx
[2/3] Compiling C++ object src/aequilibrae/paths/cython/route_choice_set_results.cpython-312-x86_64-linux-gnu.so.p/meson-generated_src_aequilibrae_paths_cython_route_choice_set_results.pyx.cpp.o
src/aequilibrae/paths/cython/route_choice_set_results.cpython-312-x86_64-linux-gnu.so.p/src/aequilibrae/paths/cython/route_choice_set_results.pyx.cpp:23628:18: warning: ‘PyObject* __pyx_f_11aequilibrae_5paths_6cython_24route_choice_set_results_binary_logit(double, double)’ defined but not used [-Wunused-function]
23628 | static PyObject *__pyx_f_11aequilibrae_5paths_6cython_24route_choice_set_results_binary_logit(double __pyx_v_min_cost, double __pyx_v_other_cost) {
      |                  ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

## Download and crop the Queensland PBF

This downloads the current Geofabrik Queensland extract, then uses Pyrosm to write a Mount Isa PBF crop. The extent matches the coordinates in the osmium example below; Pyrosm expects `[west, south, east, north]`, so the two latitude values are ordered from south to north. The source and crop are kept in `python/data/osm` (or `data/osm` when the notebook is launched from `python`).


In [2]:
from urllib.request import urlretrieve

from pyrosm import OSM

osm_data_dir = Path.cwd() / "data" / "osm"
if not osm_data_dir.is_dir():
    osm_data_dir = Path.cwd() / "python" / "data" / "osm"
osm_data_dir.mkdir(parents=True, exist_ok=True)

geofabrik_url = "https://download.geofabrik.de/australia-oceania/australia/queensland-latest.osm.pbf"
source_pbf_path = osm_data_dir / "queensland-latest.osm.pbf"
pbf_path = osm_data_dir / "mount-isa.osm.pbf"

if not source_pbf_path.is_file():
    print(f"Downloading {geofabrik_url}")
    urlretrieve(geofabrik_url, source_pbf_path)
else:
    print(f"Using downloaded source: {source_pbf_path}")

bbox = [139.45510, -20.79793, 139.57898, -20.60647]  # west, south, east, north
print(f"Cropping Queensland PBF to {bbox}")
if not pbf_path.exists():
    OSM(source_pbf_path, bounding_box=bbox).to_pbf(
        output_path=str(pbf_path), repack=True
    )
print(f"Mount Isa PBF: {pbf_path}")

Using downloaded source: /home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/scratch/python/data/osm/queensland-latest.osm.pbf
Cropping Queensland PBF to [139.4551, -20.79793, 139.57898, -20.60647]
Mount Isa PBF: /home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/scratch/python/data/osm/mount-isa.osm.pbf


## Pre-download the OSM data

Set import_source to osmnx_cache to fetch the OSM response once into the persistent cache directory, or pbf to use a previously prepared local .osm.pbf file. The cache mode uses the same OSMnx query settings as the AequilibraE importer, so the import cell below reuses the cached response instead of making another Overpass request.


In [3]:
import_source = "pbf"  # Choose "osmnx_cache" or "pbf".

if import_source not in {"osmnx_cache", "pbf"}:
    raise ValueError("import_source must be either 'osmnx_cache' or 'pbf'")

if import_source == "pbf":
    if not pbf_path.is_file():
        raise FileNotFoundError(f"Place a Mount Isa .osm.pbf extract at: {pbf_path}")
    print(f"Using local PBF: {pbf_path}")
else:
    import geopandas as gpd
    import osmnx as ox
    from shapely.geometry import Point

    # Keep OSMnx HTTP response files in a stable, shareable folder.
    ox.settings.use_cache = True
    ox.settings.cache_folder = osm_cache_dir

    place_name = "Mount Isa, Queensland, 4825, Australia"
    latitude, longitude = ox.geocode(place_name)
    buffer_m = 5_000
    centre = gpd.GeoSeries([Point(longitude, latitude)], crs="EPSG:4326")
    local_crs = centre.estimate_utm_crs()
    model_area = centre.to_crs(local_crs).buffer(buffer_m).to_crs("EPSG:4326").iloc[0]
    print(f"Study area: {buffer_m / 1_000:g} km radius in {local_crs}")

    # Warm the Overpass response cache before AequilibraE imports the same area.
    graph = ox.graph_from_polygon(
        model_area,
        network_type="all",
        simplify=False,
        retain_all=True,
        truncate_by_edge=False,
        custom_filter=None,
    )
    print(
        f"Downloaded graph: {len(graph.nodes):,} nodes, {len(graph.edges):,} directed edges"
    )
    del graph
    print(f"OSMnx response cache: {osm_cache_dir}")


Using local PBF: /home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/scratch/python/data/osm/mount-isa.osm.pbf


## Import the cached data into AequilibraE
  With osmnx_cache, the AequilibraE importer repeats the same polygon query and OSMnx serves it from osmnx_cache. Once the cache has been prepared, this can run without reaching Nominatim or Overpass. With pbf, the importer reads the local file through pyrosm.


In [4]:
project = Project()
project.new(project_path)

if import_source == "pbf":
    project.network.importer.osm(
        pbf_path=pbf_path,
        modes=("car", "transit", "bicycle", "walk"),
        simplify=True,
    )
else:
    project.network.importer.osm(
        model_area=model_area,
        modes=("car", "transit", "bicycle", "walk"),
        simplify=False,
    )


## Inspect the imported network

The project keeps its network in the AequilibraE project database. Inspect the imported link and node tables and check the modes represented in the network.

In [5]:
links = project.network.links.data
nodes = project.network.nodes.data
print(f"Links: {len(links):,}\nNodes: {len(nodes):,}")
links.head()

Links: 3,798
Nodes: 2,528


,link_id,a_node,b_node,direction,distance,modes,link_type,name,fixed_cost_ab,fixed_cost_ba,speed_ab,speed_ba,travel_time_ab,travel_time_ba,capacity_ab,capacity_ba,other_attributes,geometry
0,1,100000,100001,0,255.113000,bctw,residential,George Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27210446"",""source_ids"":""{\""schem...","LINESTRING (139.4999 -20.722, 139.50227 -20.72..."
1,2,100000,100002,0,54.927000,bctw,residential,Mullan Street,None,None,NaN,NaN,None,None,None,None,"{""lanes_ab"":1.0,""lanes_ba"":1.0,""source_id"":""26...","LINESTRING (139.4999 -20.722, 139.49996 -20.72..."
2,3,100000,100003,0,89.054000,bctw,residential,George Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27210446"",""source_ids"":""{\""schem...","LINESTRING (139.4999 -20.722, 139.49937 -20.72..."
3,4,100001,100004,0,112.880933,bctw,residential,George Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27210446"",""source_ids"":""{\""schem...","LINESTRING (139.50227 -20.72142, 139.50326 -20..."
4,5,100001,100005,0,109.294000,bctw,residential,King Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27247376"",""source_ids"":""{\""schem...","LINESTRING (139.50227 -20.72142, 139.50209 -20..."


In [6]:
# zones_file = Path(
#     r"D:\OuterLoop\aequilibrae_training\python\data\Mount Isa SA1.parquet"
# )
zones_file = Path("/mnt/c/Users/TylerPearn/Documents/aequilibrae_training/Mount Isa SA1.parquet")


# employment_file = Path(
#     r"D:\OuterLoop\aequilibrae_training\python\data\Mount Isa SA1 employment.csv"
# )
employment_file = Path("/mnt/c/Users/TylerPearn/Documents/aequilibrae_training/Mount Isa SA1 employment.csv")


In [7]:
zones = gpd.read_parquet(zones_file)[["SA1_CODE_2021", "geometry"]].explode()
zones = zones.assign(zone_id=range(1, len(zones) + 1))

zones["SA1_CODE_2021"] = zones["SA1_CODE_2021"].astype(int)
zones = zones.set_index("SA1_CODE_2021")
zones.head()

,geometry,zone_id
SA1_CODE_2021,,
31502140501,"POLYGON ((139.49615 -20.7372, 139.49527 -20.73...",1
31502140502,"POLYGON ((139.48816 -20.70872, 139.48813 -20.7...",2
31502140503,"POLYGON ((139.49131 -20.73395, 139.49142 -20.7...",3
31502140504,"POLYGON ((139.4904 -20.72597, 139.49034 -20.72...",4
31502140505,"POLYGON ((139.49807 -20.7245, 139.49799 -20.72...",5


In [8]:
employment = (
    pd.read_csv(employment_file, skiprows=9)  # Skip ABS header
    .tail(-1)  # Drop the first row which is blank for the two row header
    .head(-5)  # Drop the last 5 rows which contain data source info
    .loc[:, :"Total"]  # Drop columns after "Total", just whitespace
    .rename(
        columns={"1-digit level INDP Industry of Employment": "SA1_CODE_2021"}
    )  # Pandas reads this column name incorrectly due to the multi-row header
    .astype(int)
).set_index("SA1_CODE_2021")

column_names = {
    column: column.lower().replace(" ", "_").replace(",", "")
    for column in employment.columns
} | {"SA1_CODE_2021": "SA1_CODE_2021"}

employment = employment.rename(columns=column_names)

employment.head()

,agriculture_forestry_and_fishing,mining,manufacturing,electricity_gas_water_and_waste_services,construction,wholesale_trade,retail_trade,accommodation_and_food_services,transport_postal_and_warehousing,information_media_and_telecommunications,...,administrative_and_support_services,public_administration_and_safety,education_and_training,health_care_and_social_assistance,arts_and_recreation_services,other_services,inadequately_described,not_stated,not_applicable,total
SA1_CODE_2021,,,,,,,,,,,,,,,,,,,,,
31502140601,20,22,6,5,13,4,10,7,25,0,...,3,15,7,28,4,12,0,5,175,361
31502140602,4,36,5,3,7,0,9,10,27,0,...,3,17,14,18,0,0,0,4,145,311
31502140603,10,35,4,0,17,0,21,23,31,0,...,10,27,9,16,0,14,0,12,295,519
31502140604,4,25,10,0,25,4,12,3,18,0,...,3,17,23,5,0,5,0,3,153,326
31502140605,5,0,0,0,9,0,3,7,3,0,...,0,6,10,4,0,3,0,0,107,152


In [9]:
zones = zones.join(employment, how="left")

In [10]:
fields = project.network.zones.fields
existing_fields = fields.all_fields()

for proper_name, column in column_names.items():
    if column not in existing_fields:
        fields.add(column, description=f"ABS: {proper_name}", data_type="integer")
        print("Added field:", column)

fields.save()

Added field: agriculture_forestry_and_fishing
Added field: mining
Added field: manufacturing
Added field: electricity_gas_water_and_waste_services
Added field: construction
Added field: wholesale_trade
Added field: retail_trade
Added field: accommodation_and_food_services
Added field: transport_postal_and_warehousing
Added field: information_media_and_telecommunications
Added field: financial_and_insurance_services
Added field: rental_hiring_and_real_estate_services
Added field: professional_scientific_and_technical_services
Added field: administrative_and_support_services
Added field: public_administration_and_safety
Added field: education_and_training
Added field: health_care_and_social_assistance
Added field: arts_and_recreation_services
Added field: other_services
Added field: inadequately_described
Added field: not_stated
Added field: not_applicable
Added field: total
Added field: SA1_CODE_2021


In [11]:
centroid_ids = project.network.zones.insert_from(zones.reset_index())

In [12]:
project.network.zones.add_centroids()

Adding centroids                                  :   0%|          | 0/52 [00:00<?, ?it/s]

In [13]:
project.network.zones.connect_mode("b")

Connecting zones                                  :   0%|          | 0/52 [00:00<?, ?it/s]

In [14]:
from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap
import shapely
from shapely.geometry import box
import geopandas as gpd
from matplotlib import colormaps
import numpy as np

In [15]:
from aequilibrae.paths.route_choice import RouteChoice

In [16]:
# import numpy as np

In [17]:
project.network.build_graphs(modes=["b"])
graph = project.network.graphs["b"]
graph.network["utility"] = 0.01 / 1000 * graph.network["distance"]

graph.prepare_graph(np.asarray(centroid_ids))
graph.set_graph("utility")

rc = RouteChoice(graph)
rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)

/home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/src/aequilibrae/paths/graph.py:263: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
/home/tylerpearn/outerl_code/my_aequilibrae_fork/aequilibrae/src/aequilibrae/paths/graph.py:263: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the in

In [18]:
rc.prepare([(15, 49)])

In [19]:
rc.execute()

0/1 ODs processed                                 :   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
routes = rc.get_results()

In [21]:
routes

,origin id,destination id,cost,mask,path overlap,probability,route set
0,15,49,0.102674,1,0.261603,0.140401,"[3813, -3786, -3783, -1412, -1405, -1400, -139..."
1,15,49,0.137967,1,0.851607,0.441204,"[3813, -3786, -3783, -1412, -1405, 1404, 1410,..."
2,15,49,0.102857,1,0.283818,0.152296,"[3813, -3786, -3783, -1412, -1405, -1400, -139..."
3,15,49,0.102820,1,0.251192,0.134794,"[3813, -3786, -3783, 3784, -1413, -1405, -1400..."
4,15,49,0.102634,1,0.244644,0.131304,"[3813, -3786, -3783, -1412, -1405, -1400, -139..."


In [22]:
links = project.network.links.data
nodes = project.network.nodes.data

In [23]:
from functools import reduce

In [24]:
route_links_ids = reduce(np.union1d, routes["route set"].abs())
route_links = links[links["link_id"].isin(route_links_ids)]

separate_routes_ids = list(routes["route set"].abs())
# [route_ids for route_ids in routes["route set"].abs()]
separate_routes_ids

[array([3813, 3786, 3783, 1412, 1405, 1400, 1396, 1392, 1390, 1380, 1374,
        1357, 1354, 1350, 1345, 1342, 1259, 1258, 1260,  962,  957,  122,
         121,  971,  999, 1004,  781,  779, 1002,  791, 2388, 1684,  728,
         326,  693,  678,  679,  694,  683,  682, 1679, 1677, 1671, 1672,
         631,  629,  630,  620,  616,  612,  610,  606,  602,  597,  362,
         359, 3847]),
 array([3813, 3786, 3783, 1412, 1405, 1404, 1410, 1419, 1421, 1425, 1269,
        1266, 1264, 1267, 1271, 1313, 1290, 1285, 1284, 1288, 1292, 1307,
        1303, 1300, 1296, 1294, 1295, 1298, 1316, 1256, 1250, 1245, 1244,
         276,  269,  258,  259,  270, 1133, 1132, 1139, 1153, 1199, 1192,
        1179, 1169, 1165, 1120,  913,  911,  910, 1115, 1110,  845,  840,
         839,  863, 2205,  718,  716, 2392, 2390, 2244,  614,  613,  610,
         606,  602,  597,  362,  359, 3847]),
 array([3813, 3786, 3783, 1412, 1405, 1400, 1396, 1392, 1390, 1380, 1374,
        1357, 1354, 1350, 1345, 1342, 1259, 

In [25]:
route_links_ids

array([ 121,  122,  258,  259,  269,  270,  276,  326,  359,  362,  597,
        602,  606,  610,  612,  613,  614,  616,  620,  629,  630,  631,
        678,  679,  682,  683,  693,  694,  716,  718,  728,  779,  781,
        791,  839,  840,  845,  863,  910,  911,  913,  957,  961,  962,
        969,  971,  972,  978,  983,  999, 1002, 1004, 1110, 1115, 1120,
       1132, 1133, 1139, 1153, 1165, 1169, 1179, 1192, 1199, 1244, 1245,
       1250, 1256, 1258, 1259, 1260, 1264, 1266, 1267, 1269, 1271, 1284,
       1285, 1288, 1290, 1292, 1294, 1295, 1296, 1298, 1300, 1303, 1307,
       1313, 1316, 1342, 1345, 1350, 1354, 1357, 1374, 1380, 1390, 1392,
       1396, 1400, 1404, 1405, 1410, 1412, 1413, 1419, 1421, 1425, 1671,
       1672, 1677, 1679, 1684, 2205, 2244, 2388, 2390, 2392, 3783, 3784,
       3786, 3813, 3847])

In [26]:
hover_links = links[
    [
        "link_id",
        "link_type",
        "direction",
        "geometry",
    ]
].rename(
    columns={
        "link_id": "Link ID",
        "link_type": "Link type",
        "direction": "Direction",
    }
)

link_layer = PathLayer.from_geopandas(
    hover_links,
    width_units="pixels",
    width_min_pixels=1,
    width_max_pixels=8,
    auto_highlight=True,
)
rc_layer = PathLayer.from_geopandas(
    route_links[["geometry"]],
    width_units="pixels",
    width_min_pixels=4,
    width_max_pixels=128,
    get_color=[0, 0, 255],
    pickable=False,
)

nodes_layer = ScatterplotLayer.from_geopandas(
    nodes[nodes["is_centroid"] != 1][
        [
            "node_id",
            "geometry",
        ]
    ],
    get_fill_color=[255, 0, 0],
    get_radius=10,
)
centroid_layer = ScatterplotLayer.from_geopandas(
    nodes[nodes["is_centroid"] == 1][
        [
            "node_id",
            "geometry",
        ]
    ],
    get_fill_color=[0, 255, 0],
    get_radius=20,
)

# 139.45510,-20.60647,139.57898,-20.79793
bbox = PolygonLayer.from_geopandas(
    gpd.GeoDataFrame(
        geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"
    ),
    get_fill_color=[100, 100, 100, 128],
)

traffic_map = Map(
    layers=[bbox, link_layer, rc_layer, nodes_layer, centroid_layer],
    basemap=None,
    height="1200px",
    show_side_panel=False,
    show_tooltip=True,
)
traffic_map

In [ ]:
# graph.prepare_graph(np.asarray(centroid_ids))
# graph.set_graph("distance")

# rc = RouteChoice(graph)
# rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)

def make_pretty_route_choice(source, dest, rc: RouteChoice, routes, project: Project, separate_routes: bool = True):
    routes = routes[not routes["mask"] != 1]
    routes = routes.sort_values(by='cost', ascending=False) # sort descending
    links = project.network.links.data
    nodes = project.network.nodes.data

    route_links_ids = reduce(np.union1d, routes["route set"].abs())
    route_links = links[links["link_id"].isin(route_links_ids)]
    separate_routes_ids = list(routes["route set"].abs())
    separate_routes_links = [links[links["link_id"].isin(ids)] for ids in separate_routes_ids]


    hover_links = links[
        [
            "link_id",
            "link_type",
            "direction",
            "geometry",
        ]
    ].rename(
        columns={
            "link_id": "Link ID",
            "link_type": "Link type",
            "direction": "Direction",
        }
    )

    link_layer = PathLayer.from_geopandas(
        hover_links,
        width_units="pixels",
        width_min_pixels=1,
        width_max_pixels=8,
        auto_highlight=True,
    )

    n_routes = len(separate_routes_links)
    normalized = np.linspace(0, 1, n_routes)
    turbo_colors = apply_continuous_cmap(normalized, colormaps["turbo"] )
    # turbo_colors is an array of shape (n_routes, 4), values 0-255
    MIN_WIDTH, MAX_WIDTH = 4, 12  # pixels
    widths = np.linspace(MAX_WIDTH, MIN_WIDTH, n_routes)

    rc_layers: list[PathLayer] = []
    if separate_routes:
        for i, (single_route_links, color, width) in enumerate(zip(separate_routes_links, turbo_colors, widths)):
            rc_layer = PathLayer.from_geopandas(
                single_route_links[["geometry"]],
                width_units="pixels",
                get_width=float(width),
                width_min_pixels=1,
                width_max_pixels=MAX_WIDTH,
                get_color=color.tolist(),
                pickable=False,
            )
            rc_layers.append(rc_layer)
    else:
        rc_layer = PathLayer.from_geopandas(
            route_links[["geometry"]],
            width_units="pixels",
            width_min_pixels=4,
            width_max_pixels=128,
            get_color=[0, 0, 255],
            pickable=False,
        )
        rc_layers.append(rc_layer)

    nodes_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] != 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[255, 0, 0],
        get_radius=10,
    )
    centroid_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] == 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[0, 255, 0],
        get_radius=20,
    )

    # 139.45510,-20.60647,139.57898,-20.79793
    bbox = PolygonLayer.from_geopandas(
        gpd.GeoDataFrame(
            geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"
        ),
        get_fill_color=[100, 100, 100, 128],
    )

    traffic_map = Map(
        layers=[bbox, link_layer, *rc_layers, nodes_layer, centroid_layer],
        basemap=None,
        height="1200px",
        show_side_panel=False,
        show_tooltip=True,
    )
    return traffic_map

In [28]:
def inverse_binary_logit(prob, beta0, beta1):
    if prob == 1.0:
        return float("inf")
    elif prob == 0.0:
        return -float("inf")
    else:
        return (np.log(prob / (1.0 - prob)) - beta0) / beta1

In [32]:
# graph.prepare_graph(np.asarray(centroid_ids))
# graph.set_graph("distance")

low_cutoff_rc = RouteChoice(graph)
my_cutoff = 0.0
low_cutoff_rc.set_choice_set_generation(algorithm="LP", max_routes=5, max_depth=5, cutoff_prob = my_cutoff)

low_cutoff_rc.prepare([(14, 50)])
low_cutoff_rc.execute(perform_assignment=False) # route choice sets only
routes = low_cutoff_rc.get_results()

routes






0/1 ODs processed                                 :   0%|          | 0/1 [00:00<?, ?it/s]

,origin id,destination id,route set
0,14,50,"[3812, -1984, -1837, -1831, 1834, -1826, 1825,..."
1,14,50,"[3812, -1984, -1837, -1831, 1834, -1826, 1825,..."
2,14,50,"[3812, -1984, -1837, -1831, -1827, -1822, -181..."
3,14,50,"[3812, -1984, -1837, -1831, 1834, -1826, 1825,..."


In [33]:
inverse_logits = inverse_binary_logit((0.95/2) + 0.5, 0, 1)
inverse_logits

np.float64(3.6635616461296454)

In [34]:
make_pretty_route_choice(14, 50, low_cutoff_rc, routes, project, separate_routes=True)

KeyError: 'mask'

In [ ]:
my_rc = RouteChoice(graph)
my_rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)

make_pretty_route_choice(my_rc, project)

In [ ]:
# Close the project database connection when you are finished.
# project.close()